# Run independent COMPASS GAM lab analysis (R environment)

This notebook consumes cohort artifacts created by the main Python pipeline and keeps every GAM-derived feature and result under a separate `survival_analysis/GAM` output tree.

Required order:

1. Run `01_preprocessing.ipynb` through **Build prediction inputs**. Its `aggregated_landmark*.csv`, long-form lab tables, canonical-lab table, splits, and manifest are the read-only cohort inputs here.
2. Run **Stage A** here to fit lab trajectories and write GAM-only features.
3. Run **Stage B** here to select and analyze only those GAM features. No main-model rerun or Python handoff is required.

The main survival analysis continues to use only its aggregate lab statistics (`mean`, `last`, `min`, `max`, `delta`, etc.).


In [ ]:
suppressPackageStartupMessages(library(data.table))

PROJECT_ROOT <- "/data/gusev/USERS/jpconnor/code/CAIA"
GAM_DIR <- file.path(PROJECT_ROOT, "COMPASS", "survival_analysis", "GAM")

# Choose "profile_data" for the merged Parquets or "baseline" for ALL_2025_03.
DATA_VARIANT <- "profile_data"
ARM <- "adt"
LANDMARK_DAYS <- c(0L, 90L, 180L)
FORCE_RERUN <- TRUE
MIN_PATIENT_COVERAGE <- 0.20
N_WORKERS <- 8L

DATA_ROOT <- switch(
  DATA_VARIANT,
  profile_data = "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS_PROFILE_DATA",
  baseline = "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS",
  stop(sprintf("Unknown DATA_VARIANT: %s", DATA_VARIANT))
)
COHORT_INPUTS_DIR <- file.path(DATA_ROOT, "survival_analysis", sprintf("prediction_inputs_%s", ARM))
GAM_OUTPUT_DIR <- file.path(DATA_ROOT, "survival_analysis", "GAM", ARM)
FEATURE_OUTPUT_DIR <- file.path(GAM_OUTPUT_DIR, "features")
NONLINEAR_OUTPUT_DIR <- file.path(GAM_OUTPUT_DIR, "nonlinearity")
RSCRIPT <- file.path(R.home("bin"), "Rscript")

run_r_script <- function(script_name, args, wait = TRUE) {
  script_path <- file.path(GAM_DIR, script_name)
  if (!file.exists(script_path)) stop(sprintf("Missing R script: %s", script_path))
  command_args <- shQuote(c(script_path, args))
  cat(sprintf("[run ] %s %s\n", RSCRIPT, paste(command_args, collapse = " ")))
  status <- system2(RSCRIPT, args = command_args, stdout = "", stderr = "", wait = wait)
  if (wait && !identical(status, 0L)) stop(sprintf("%s failed with status %s", script_name, status))
  invisible(status)
}

wait_for_pid <- function(pid) {
  system2("bash", args = c("-c", shQuote(sprintf("while kill -0 %d 2>/dev/null; do sleep 1; done", pid))))
  invisible(NULL)
}

cat(sprintf("R:                 %s\n", R.version.string))
cat(sprintf("cohort inputs:     %s\n", COHORT_INPUTS_DIR))
cat(sprintf("GAM outputs:       %s\n", GAM_OUTPUT_DIR))
cat(sprintf("n workers:         %d\n", N_WORKERS))


## Stage A — hierarchical trajectory GAM features

Run this after `build_prediction_inputs` has created the pre-treatment long tables and canonical-lab file.


In [ ]:
required_inputs <- c(
  file.path(COHORT_INPUTS_DIR, "canonical_labs_train_val.csv"),
  file.path(COHORT_INPUTS_DIR, "build_manifest.json"),
  file.path(COHORT_INPUTS_DIR, sprintf("aggregated_landmark%d.csv", LANDMARK_DAYS)),
  file.path(COHORT_INPUTS_DIR, sprintf("pre_treatment_lab_long_landmark%d.csv", LANDMARK_DAYS))
)
missing_inputs <- required_inputs[!file.exists(required_inputs)]
if (length(missing_inputs)) {
  stop(sprintf("Missing main-pipeline cohort input(s):\n%s", paste(missing_inputs, collapse = "\n")))
}

dir.create(FEATURE_OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
trajectory_outputs <- file.path(FEATURE_OUTPUT_DIR, sprintf("gam_trajectory_features_landmark%d.csv", LANDMARK_DAYS))
diagnostic_outputs <- file.path(FEATURE_OUTPUT_DIR, sprintf("gam_fit_diagnostics_landmark%d.csv", LANDMARK_DAYS))
curve_outputs <- file.path(FEATURE_OUTPUT_DIR, sprintf("gam_trajectory_curves_landmark%d.csv", LANDMARK_DAYS))
stage_a_outputs <- c(trajectory_outputs, diagnostic_outputs, curve_outputs)

if (FORCE_RERUN || !all(file.exists(stage_a_outputs))) {
  stage_a_timing <- system.time(
    run_r_script(
      "gam_trajectory_features.R",
      c(
        "--inputs-dir", COHORT_INPUTS_DIR,
        "--output-dir", FEATURE_OUTPUT_DIR,
        "--landmark-days", paste(LANDMARK_DAYS, collapse = ","),
        "--k-pop", "16", "--k-pat", "8",
        "--max-fs-patients", "0", "--patient-ridge-lambda", "5",
        "--trailing-window-days", "180", "--nthreads", "1",
        "--fit-split", "all", "--n-workers", as.character(N_WORKERS),
        "--curve-labs", "PSA,Testosterone",
        "--curve-grid-points", "9", "--auc-grid-points", "9"
      )
    )
  )
  cat(sprintf("[timing] Stage A wall time: %.1fs\n", stage_a_timing[["elapsed"]]))
} else {
  cat("[skip] all GAM trajectory outputs already exist\n")
}
stopifnot(all(file.exists(stage_a_outputs)))

diagnostics_all <- rbindlist(lapply(diagnostic_outputs, fread))
print(diagnostics_all[, .N, by = basis_used])
cat(sprintf("[timing] sum(fit_seconds): %.1fs; median edf_pop: %.2f\n",
            sum(diagnostics_all$fit_seconds, na.rm = TRUE),
            median(diagnostics_all$edf_pop, na.rm = TRUE)))


## Stage B — GAM-only Cox analysis

This stage joins the GAM feature table to outcome/observation-count columns from the main cohort, applies its own train/validation coverage gate, and fits linear and smooth Cox effects. It never reads the main analysis feature-selection output.


In [ ]:
dir.create(NONLINEAR_OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
nonlinear_outputs <- file.path(NONLINEAR_OUTPUT_DIR, sprintf("gam_cox_nonlinearity_landmark%d.csv", LANDMARK_DAYS))

stage_b_timing <- system.time({
  for (i in seq_along(LANDMARK_DAYS)) {
    landmark <- LANDMARK_DAYS[[i]]
    if (!FORCE_RERUN && file.exists(nonlinear_outputs[[i]])) {
      cat(sprintf("[skip] landmark +%dd GAM Cox output exists\n", landmark))
      next
    }
    run_r_script(
      "gam_cox_nonlinearity.R",
      c(
        "--inputs-dir", COHORT_INPUTS_DIR,
        "--gam-features-dir", FEATURE_OUTPUT_DIR,
        "--output-dir", NONLINEAR_OUTPUT_DIR,
        "--landmark-days", as.character(landmark),
        "--min-patient-coverage", as.character(MIN_PATIENT_COVERAGE),
        "--n-workers", as.character(N_WORKERS)
      )
    )
  }
})
cat(sprintf("[timing] Stage B wall time: %.1fs\n", stage_b_timing[["elapsed"]]))
stopifnot(all(file.exists(nonlinear_outputs)))

selection_outputs <- file.path(NONLINEAR_OUTPUT_DIR, sprintf("gam_feature_selection_landmark%d.csv", LANDMARK_DAYS))
stopifnot(all(file.exists(selection_outputs)))

## GAM association summaries


In [ ]:
gam_results <- rbindlist(lapply(nonlinear_outputs, fread), fill = TRUE)

linear_hits <- gam_results[
  !is.na(q_linear),
  .(landmark_days, feature, coef_linear, p_linear, q_linear)
][order(q_linear)]
cat("Top linear GAM-feature associations:\n")
print(head(linear_hits, 25))

nonlinear_hits <- gam_results[
  !is.na(q_lrt) & q_lrt < 0.05 & edf > 1.5,
  .(landmark_days, feature, edf, p_lrt, q_lrt, delta_aic)
][order(q_lrt)]
cat("Potential nonlinear hazard relationships:\n")
print(nonlinear_hits)


## GAM figures

Two figures, written to `GAM_OUTPUT_DIR/figures/`: the fitted trajectory
curves themselves (Stage A output, per `--curve-labs` lab), and a volcano
summary of the Stage B Cox associations. Uses the same `theme_fig()`
typography as the main figure notebook, kept self-contained here since this
notebook does not source `COMPASS_generate_figures_pipeline.R`.

In [ ]:
suppressPackageStartupMessages({
  library(ggplot2)
  library(scales)
  library(ggrepel)
})

FIG_DIR <- file.path(GAM_OUTPUT_DIR, "figures")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)

# Matches theme_fig() / COLOR_PLATINUM_* in COMPASS_generate_figures_pipeline.R
# so GAM figures read consistently with the main figure set.
theme_fig <- function(base_size = 11) {
  theme_classic(base_size = base_size) +
    theme(
      plot.title        = element_text(face = "bold", size = base_size + 1),
      plot.subtitle     = element_text(size = base_size - 1, color = "#52514e"),
      axis.title        = element_text(size = base_size),
      axis.text         = element_text(size = base_size - 1.5, color = "#1a1a1a"),
      legend.key        = element_blank(),
      legend.background = element_blank(),
      legend.text       = element_text(size = base_size - 2),
      strip.background  = element_blank(),
      strip.text        = element_text(face = "bold", size = base_size - 0.5),
      axis.line         = element_line(linewidth = 0.5),
      axis.ticks        = element_line(linewidth = 0.5),
      plot.background   = element_rect(fill = "white", color = NA),
      panel.background  = element_rect(fill = "white", color = NA),
      plot.margin       = margin(6, 8, 6, 6)
    )
}
theme_set(theme_fig())

COLOR_PLATINUM_POS <- "#2a78d6"  # blue
COLOR_NEUTRAL_INK  <- "#52514e"
COLOR_MEAN_CURVE   <- "#8e1c2b"  # matches HIGHLIGHT_COLOR in the main figure pipeline
SAVE_DPI <- 600
HAS_RAGG <- requireNamespace("ragg", quietly = TRUE)

save_fig <- function(plot, stem, width, height) {
  png_out <- file.path(FIG_DIR, paste0(stem, ".png"))
  if (HAS_RAGG) {
    ggsave(png_out, plot = plot, width = width, height = height, units = "in",
           dpi = SAVE_DPI, bg = "white", device = ragg::agg_png)
  } else {
    ggsave(png_out, plot = plot, width = width, height = height, units = "in",
           dpi = SAVE_DPI, bg = "white")
  }
  cat(sprintf("[fig ] %s\n", png_out))
  invisible(png_out)
}

cat(sprintf("figure output dir: %s\n", FIG_DIR))

### Fitted trajectory curves

Per-patient GAM-fitted curves (thin, translucent) with the population mean
overlaid (bold), on the trailing pre-landmark window. One panel per lab per
landmark. Patient identity carries no meaning at this N, so individual curves
share one neutral color; only the mean curve is highlighted.

In [ ]:
curves_all <- rbindlist(
  Map(function(path, lm) { d <- fread(path); d[, landmark_days := lm]; d },
      curve_outputs, LANDMARK_DAYS),
  use.names = TRUE
)

if (nrow(curves_all) == 0) {
  cat("[skip] no fitted curve points written (--curve-labs empty or write-curves=false)\n")
} else {
  curves_all[, landmark_label := sprintf("landmark +%dd", landmark_days)]
  curves_all[, landmark_label := factor(landmark_label,
    levels = sprintf("landmark +%dd", sort(LANDMARK_DAYS)))]

  mean_curves <- curves_all[, .(GAM_FITTED = mean(GAM_FITTED, na.rm = TRUE)),
                            by = .(LAB_NAME, landmark_label, t_lab)]

  p_curves <- ggplot(curves_all, aes(t_lab, GAM_FITTED)) +
    geom_line(aes(group = .data[[id_col]]), color = COLOR_NEUTRAL_INK,
              alpha = 0.06, linewidth = 0.3) +
    geom_line(data = mean_curves, color = COLOR_MEAN_CURVE, linewidth = 1.1) +
    facet_grid(LAB_NAME ~ landmark_label, scales = "free") +
    labs(x = "days relative to landmark", y = "GAM-fitted lab value",
         title = "Fitted GAM trajectory curves",
         subtitle = sprintf("%s / %s -- thin lines: per-patient fit; bold: population mean",
                            toupper(DATA_VARIANT), toupper(ARM))) +
    theme_fig()

  n_labs <- uniqueN(curves_all$LAB_NAME)
  save_fig(p_curves, "gam_trajectory_curves", width = 3.2 * length(LANDMARK_DAYS), height = 2.6 * n_labs)
  print(p_curves)
}

### Cox association volcano

Linear GAM-feature associations with the platinum hazard, colored by
significance status. Points also flagged nonlinear (`q_lrt < 0.05` and
`edf > 1.5`, the same threshold used in the association summary above) are
outlined. Same visual language as Figure 3's univariate volcano in the main
figure notebook.

In [ ]:
volcano_dt <- gam_results[!is.na(p_linear) & !is.na(coef_linear)]

if (nrow(volcano_dt) == 0) {
  cat("[skip] no linear GAM-Cox associations to plot\n")
} else {
  volcano_dt[, neglog10p := -log10(p_linear)]
  volcano_dt[, is_nonlinear := !is.na(q_lrt) & q_lrt < 0.05 & edf > 1.5]
  volcano_dt[, status := fifelse(!is.na(q_linear) & q_linear < 0.05,
                                 "Significant (q<0.05)", "Not significant")]
  volcano_dt[, status := factor(status, levels = c("Not significant", "Significant (q<0.05)"))]
  volcano_dt[, landmark_label := factor(sprintf("landmark +%dd", landmark_days),
                                        levels = sprintf("landmark +%dd", sort(LANDMARK_DAYS)))]

  NS_COLOR <- "#c9c6bf"
  status_colors <- c(`Not significant` = NS_COLOR, `Significant (q<0.05)` = COLOR_PLATINUM_POS)

  p_volcano <- ggplot(volcano_dt, aes(coef_linear, neglog10p)) +
    geom_point(aes(color = status), size = 2.2, alpha = 0.85) +
    geom_point(data = volcano_dt[is_nonlinear == TRUE],
               shape = 21, color = COLOR_MEAN_CURVE, fill = NA,
               size = 3.6, stroke = 0.9) +
    ggrepel::geom_text_repel(
      data = volcano_dt[status == "Significant (q<0.05)"],
      aes(label = feature), size = 3, max.overlaps = 15,
      segment.color = "#95a5a6", segment.size = 0.3, min.segment.length = 0
    ) +
    facet_wrap(~landmark_label, nrow = 1) +
    scale_color_manual(values = status_colors, name = NULL) +
    labs(x = "GAM feature Cox coefficient (linear)", y = expression(-log[10](p)),
         title = "GAM-feature Cox associations",
         subtitle = sprintf("%s / %s -- outlined points: nonlinear (q_lrt<0.05, edf>1.5)",
                            toupper(DATA_VARIANT), toupper(ARM))) +
    theme_fig() +
    theme(legend.position = "bottom")

  save_fig(p_volcano, "gam_cox_volcano", width = 3.6 * length(LANDMARK_DAYS), height = 4.5)
  print(p_volcano)
}